In [1]:
import os
import pandas as pd
import numpy as np
from typing import Dict, Optional, Tuple, List, Any
import logging
from datetime import datetime, timedelta
from sklearn.preprocessing import RobustScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, MultiHeadAttention, LayerNormalization, Dropout,
    Input, Concatenate, Lambda, Add, Conv1D, LSTM, GRU,
    BatchNormalization, GlobalAveragePooling1D, Multiply, Activation
)
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, TensorBoard
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l1_l2
import tensorflow as tf
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.cluster import KMeans, DBSCAN
import talib
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor
import warnings
from tqdm import tqdm

@dataclass
class ModelConfig:
    """Configuração do modelo otimizada para multi-GPU"""
    seq_len: int = 60
    num_heads: int = 4
    head_size: int = 128
    ff_dim: int = 256
    dropout: float = 0.1
    learning_rate: float = 0.001
    batch_size: int = 128  # Aumentado para multi-GPU
    epochs: int = 200
    validation_split: float = 0.2
    early_stopping_patience: int = 20
    reduce_lr_patience: int = 10
    min_delta: float = 1e-5
    num_layers: int = 3
    hidden_size: int = 128
    attention_size: int = 64

@dataclass
class DataConfig:
    """Configuração para análise WDO"""
    volume_threshold: float = 0.7
    price_cluster_threshold: float = 0.015
    min_touches: int = 4
    outlier_std: float = 3.0
    min_volume_percentile: float = 0.1
    correlation_window: int = 20
    volatility_window: int = 20
    momentum_window: int = 10

class B3DataProcessor:
    """Processador de dados B3"""
    def __init__(self):
        self.setup_logging()
    
    def setup_logging(self):
        self.logger = logging.getLogger(__name__)
    
    def load_and_process_file(self, file_path: str) -> pd.DataFrame:
        """Carrega e processa arquivo da B3 com encoding UTF-8 e renomeação robusta"""
        try:
            # Carrega dados com encoding UTF-8 para lidar com caracteres especiais
            df = pd.read_csv(file_path, encoding="utf-8")
            
            # Limpa nomes de colunas, removendo pontos, espaços e convertendo para maiúsculas
            df.columns = df.columns.str.replace('.', '', regex=False).str.replace(' ', '_').str.upper()
            
            # Mapeamento manual de colunas para garantir nomes corretos
            rename_map = {
                'NUM_NEGOC': 'NUM_NEGOC',
                'PRECO_ABERT': 'PRECO_ABERT',
                'PRECO_MIN': 'PRECO_MIN',
                'PRECO_MAX': 'PRECO_MAX',
                'PRECO_MED': 'PRECO_MED',
                'ULT_PRECO': 'ULT_PRECO',
                'AJUSTE': 'AJUSTE',
                'VAR_PTOS': 'VAR_PTOS',
                'ULT_OF_COMPRA': 'ULT_OF_COMPRA',
                'ULT_OF_VENDA': 'ULT_OF_VENDA',
                'VOL': 'VOL'
            }
            
            # Aplica o mapeamento de renomeação
            df.rename(columns=rename_map, inplace=True)
            
            # Verificação de presença das colunas essenciais após renomeação
            required_columns = ['VOL', 'PRECO_ABERT', 'ULT_PRECO', 'PRECO_MIN', 'PRECO_MAX']
            missing_cols = [col for col in required_columns if col not in df.columns]
            if missing_cols:
                self.logger.warning(f"Colunas necessárias não encontradas após renomeação: {missing_cols}")
                return pd.DataFrame()  # Retorna vazio se faltam colunas importantes
                
            # Executa o processamento do arquivo WDO
            return self._process_wdo(df, file_path)
                        
        except Exception as e:
            self.logger.error(f"Erro ao carregar {file_path}: {str(e)}")
            return pd.DataFrame()

    
    def _clean_numeric(self, value: Any) -> float:
        """Limpa valores numéricos da B3"""
        try:
            if pd.isna(value) or value == '' or str(value).strip() in ['0000', '00000', '-']:
                return 0.0
            
            if isinstance(value, (int, float)):
                return float(value)
            
            if isinstance(value, str):
                # Remove sufixos +/-
                value = value.rstrip('+-')
                # Remove pontos de milhar e converte vírgula
                value = value.replace('.', '').replace(',', '.')
                return float(value)
                
            return 0.0
            
        except Exception:
            return 0.0
           
    
    def _process_wdo(self, df: pd.DataFrame, file_path: str) -> pd.DataFrame:
        """Processa arquivo WDO com validação melhorada e colunas corrigidas"""
        try:
            self.logger.debug(f"Processando arquivo: {file_path}")
            
            # Extrai a data do arquivo
            file_date = os.path.basename(file_path).split('_')[1].split('.')[0]
            date = pd.to_datetime(file_date, format='%d%m%Y')
            
            # Limpa volume e preços
            df['VOL'] = df.get('VOL', df.get('VOL.')).apply(self._clean_numeric)
            df['PRECO_ABERT'] = df.get('PRECO_ABERT', df.get('PREÇO ABERT.')).apply(self._clean_numeric)
            df['PRECO_MIN'] = df.get('PRECO_MIN', df.get('PREÇO MÍN.')).apply(self._clean_numeric)
            df['PRECO_MAX'] = df.get('PRECO_MAX', df.get('PREÇO MÁX.')).apply(self._clean_numeric)
            df['PRECO_MED'] = df.get('PRECO_MED', df.get('PREÇO MÉD.')).apply(self._clean_numeric)
            df['ULT_PRECO'] = df.get('ULT_PRECO', df.get('ÚLT. PREÇO')).apply(self._clean_numeric)
            df['AJUSTE'] = df.get('AJUSTE', df.get('AJUSTE')).apply(self._clean_numeric)
            
            # Filtra linhas com volume válido
            df_valid = df[df['VOL'] > 0].copy()
            if df_valid.empty:
                self.logger.debug(f"Nenhum volume válido encontrado em {file_path}")
                return pd.DataFrame()
    
            # Processa dados do vencimento mais líquido
            max_vol_idx = df_valid['VOL'].idxmax()
            row = df_valid.loc[max_vol_idx]
            
            result = {
                'date': date,
                'volume': row['VOL'],
                'abertura': row['PRECO_ABERT'],
                'minimo': row['PRECO_MIN'],
                'maximo': row['PRECO_MAX'],
                'medio': row['PRECO_MED'],
                'ultimo': row['ULT_PRECO'],
                'ajuste': row['AJUSTE']
            }
            
            if pd.isna(result['ultimo']) or pd.isna(result['volume']):
                self.logger.debug(f"Preços ou volume inválidos em {file_path}")
                return pd.DataFrame()
            
            final_df = pd.DataFrame([result])
            final_df['range'] = final_df['maximo'] - final_df['minimo']
            final_df['body'] = final_df['ultimo'] - final_df['abertura']
            
            return final_df
    
        except Exception as e:
            self.logger.error(f"Erro no processamento WDO: {file_path} - {str(e)}")
            return pd.DataFrame()
    
class WDOAnalysis:
    def __init__(self, base_path: str, cache_path: str):
        self.base_path = base_path
        self.cache_path = cache_path
        self.setup_paths()
        self.setup_logging()
        self.setup_gpu()
        
        self.data_processor = B3DataProcessor()
        self.model_config = ModelConfig()
        self.data_config = DataConfig()
    
    def setup_paths(self):
        """Configura diretórios"""
        self.ANALYSIS_PATH = os.path.join(self.cache_path, "analysis")
        self.MODEL_PATH = os.path.join(self.cache_path, "models")
        os.makedirs(self.ANALYSIS_PATH, exist_ok=True)
        os.makedirs(self.MODEL_PATH, exist_ok=True)
    
    def setup_logging(self):
        """Configura logging"""
        log_path = os.path.join(self.cache_path, "wdo_analysis.log")
        logging.basicConfig(
            level=logging.INFO,
            format='%(asctime)s - %(levelname)s - %(message)s',
            handlers=[
                logging.FileHandler(log_path),
                logging.StreamHandler()
            ]
        )
        self.logger = logging.getLogger(__name__)
    
    def setup_gpu(self):
        """Configura GPUs + NPU"""
        try:
            gpus = tf.config.list_physical_devices('GPU')
            npus = tf.config.list_physical_devices('NPU')
            
            if gpus:
                # Configura cada GPU
                for gpu in gpus:
                    tf.config.experimental.set_memory_growth(gpu, True)
                
                # Estratégia multi-GPU
                strategy = tf.distribute.MirroredStrategy()
                self.logger.info(f"GPUs Disponíveis: {len(gpus)}")
            
            if npus:
                self.logger.info(f"NPUs Disponíveis: {len(npus)}")
            
            # Otimizações
            tf.config.optimizer.set_jit(True)
            tf.config.optimizer.set_experimental_options({
                "auto_mixed_precision": True,
                "layout_optimizer": True
            })
            
        except Exception as e:
            self.logger.error(f"Erro na configuração GPU/NPU: {str(e)}")
    
    def analyze_next_session(self):
        """Analisa próxima sessão do WDO"""
        try:
            self.logger.info("\nIniciando análise WDO para próxima sessão...")
            
            # Carrega dados históricos
            df = self._load_historical_data()
            if df.empty:
                raise ValueError("Dados históricos insuficientes")
                
            # Verifica colunas necessárias
            required_cols = ['ultimo', 'maximo', 'minimo', 'abertura', 'volume', 'date']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                raise ValueError(f"Colunas ausentes no DataFrame: {missing_cols}")
            
            # Informações do último pregão
            last_date = df['date'].max()
            last_data = df[df['date'] == last_date]
            
            if last_data.empty:
                raise ValueError("Dados do último pregão não encontrados")
                
            last_close = last_data['ultimo'].iloc[0]
            
            self.logger.info(f"Data base: {last_date}")
            self.logger.info(f"Último fechamento: {last_close}")
            
            # Identifica níveis
            supports, resistances = self._identify_levels(df)
            
            # Imprime relatório
            self._print_analysis_report(last_date, last_close, supports, resistances)
            
            # Salva análise
            self._save_analysis(last_date, last_close, supports, resistances)
            
        except Exception as e:
            self.logger.error(f"Erro na análise: {str(e)}")
            self.logger.debug("Stack trace:", exc_info=True)
            raise
            
    
    def _load_historical_data(self) -> pd.DataFrame:
        """Carrega dados históricos do WDO"""
        try:
            all_data = []
            wdo_path = self.base_path
            files = sorted([f for f in os.listdir(wdo_path) if f.endswith('.csv')])
            
            total_files = len(files)
            valid_files = 0
            
            for file in tqdm(files, desc="Carregando dados WDO"):
                try:
                    file_path = os.path.join(wdo_path, file)
                    
                    # Carrega arquivo
                    df = pd.read_csv(file_path, encoding="utf-")
                    
                    # Verifica colunas necessárias
                    required_columns = [
                        'VENCTO', 'VOL.', 'PREÇO ABERT.', 'PREÇO MÍN.',
                        'PREÇO MÁX.', 'PREÇO MÉD.', 'ÚLT. PREÇO', 'AJUSTE'
                    ]
                    
                    if not all(col in df.columns for col in required_columns):
                        self.logger.debug(f"Colunas faltando em {file}: {set(required_columns) - set(df.columns)}")
                        continue
                    
                    # Processa dados
                    processed_df = self.data_processor._process_wdo(df, file_path)
                    
                    if not processed_df.empty and processed_df['ultimo'].iloc[0] > 0:
                        all_data.append(processed_df)
                        valid_files += 1
                        if valid_files % 100 == 0:
                            self.logger.info(f"Processados {valid_files} arquivos válidos...")
                    
                except Exception as e:
                    self.logger.warning(f"Erro ao processar {file}: {str(e)}")
                    continue
    
            self.logger.info(f"Arquivos processados: {valid_files}/{total_files}")
            
            if not all_data:
                raise ValueError("Nenhum dado válido encontrado")
            
            # Combina dados
            final_df = pd.concat(all_data, ignore_index=True)
            final_df = final_df.sort_values('date').reset_index(drop=True)
            
            # Log final
            self.logger.info(f"Registros carregados: {len(final_df)}")
            self.logger.info(f"Período: {final_df['date'].min()} até {final_df['date'].max()}")
            self.logger.info(f"Último preço: {final_df['ultimo'].iloc[-1]:,.2f}")
            
            return final_df
            
        except Exception as e:
            self.logger.error(f"Erro ao carregar dados históricos: {str(e)}")
            raise

            
    def _identify_levels(self, df: pd.DataFrame) -> Tuple[List[dict], List[dict]]:
        """Identifica níveis de suporte e resistência"""
        try:
            # Verifica se temos dados válidos
            if df.empty:
                raise ValueError("DataFrame vazio")
                
            # Verifica colunas necessárias
            required_cols = ['ultimo', 'maximo', 'minimo', 'abertura', 'volume', 'date']
            missing_cols = [col for col in required_cols if col not in df.columns]
            if missing_cols:
                raise ValueError(f"Colunas ausentes: {missing_cols}")
    
            # Garante que temos dados numéricos válidos
            df = df.copy()
            for col in ['ultimo', 'maximo', 'minimo', 'abertura', 'volume']:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            df = df.dropna(subset=['ultimo', 'volume'])
    
            levels = []
            last_close = df['ultimo'].iloc[-1]
            
            self.logger.info(f"Processando {len(df)} registros")
            self.logger.info(f"Último fechamento: {last_close}")
    
            # 1. Volume Profile (30% do peso)
            vol_levels = self._find_volume_levels(df)
            levels.extend(vol_levels)
            self.logger.info(f"Níveis por volume: {len(vol_levels)}")
            
            # 2. Swing Points (40% do peso)
            swing_levels = self._find_swing_levels(df)
            levels.extend(swing_levels)
            self.logger.info(f"Níveis por swing: {len(swing_levels)}")
            
            # 3. Gaps Importantes (30% do peso)
            gap_levels = self._find_gap_levels(df)
            levels.extend(gap_levels)
            self.logger.info(f"Níveis por gap: {len(gap_levels)}")
    
            # Verifica se encontramos níveis
            if not levels:
                raise ValueError("Nenhum nível identificado")
                
            # Calcula força e distância
            for level in levels:
                level.update(self._calculate_level_metrics(
                    level, last_close, df['volume'].mean()
                ))
            
            # Separa e ordena níveis
            supports = sorted(
                [l for l in levels if l['price'] < last_close],
                key=lambda x: (-x['strength'], x['distance'])
            )[:30]
            
            resistances = sorted(
                [l for l in levels if l['price'] > last_close],
                key=lambda x: (-x['strength'], x['distance'])
            )[:30]
            
            self.logger.info(f"Suportes encontrados: {len(supports)}")
            self.logger.info(f"Resistências encontradas: {len(resistances)}")
            
            return supports, resistances
            
        except Exception as e:
            self.logger.error(f"Erro na identificação de níveis: {str(e)}")
            self.logger.debug("Stack trace:", exc_info=True)
            raise
   
    
    def _find_volume_levels(self, df: pd.DataFrame) -> List[dict]:
        """Encontra níveis por Volume Profile"""
        levels = []
        try:
            # Verifica se temos dados válidos
            valid_df = df[df['ultimo'] > 0].copy()
            if len(valid_df) < 2:
                return []
    
            # Cria bins garantindo valores únicos
            unique_prices = valid_df['ultimo'].unique()
            n_bins = min(100, len(unique_prices))
            
            if n_bins < 2:
                return []
                
            price_bins = pd.qcut(valid_df['ultimo'], q=n_bins, duplicates='drop')
            volume_profile = valid_df.groupby(price_bins, observed=True)['volume'].sum()
            
            # Pega top níveis por volume
            for level in volume_profile.nlargest(20).index:
                price_data = valid_df[valid_df['ultimo'].between(level.left, level.right)]
                if len(price_data) >= self.data_config.min_touches:
                    levels.append({
                        'price': level.mid,
                        'volume': volume_profile[level],
                        'touches': len(price_data),
                        'last_test': price_data['date'].max(),
                        'type': 'volume',
                        'weight': 0.30
                    })
            
            return levels
            
        except Exception as e:
            self.logger.error(f"Erro em níveis de volume: {str(e)}")
            return []

    
    def _find_swing_levels(self, df: pd.DataFrame) -> List[dict]:
        """Encontra níveis por Swing Points"""
        levels = []
        try:
            window = 20
            
            for i in range(window, len(df)-window):
                # Swing High
                if (df['maximo'].iloc[i] > df['maximo'].iloc[i-window:i].max() and 
                    df['maximo'].iloc[i] > df['maximo'].iloc[i+1:i+window+1].max()):
                    touches = sum(abs(df['maximo'] - df['maximo'].iloc[i]) < 
                                df['maximo'].iloc[i] * self.data_config.price_cluster_threshold)
                    if touches >= self.data_config.min_touches:
                        levels.append({
                            'price': df['maximo'].iloc[i],
                            'volume': df['volume'].iloc[i],
                            'touches': touches,
                            'last_test': df['date'].iloc[i:].max(),
                            'type': 'swing_high',
                            'weight': 0.40  # Peso do método
                        })
                
                # Swing Low
                if (df['minimo'].iloc[i] < df['minimo'].iloc[i-window:i].min() and 
                    df['minimo'].iloc[i] < df['minimo'].iloc[i+1:i+window+1].min()):
                    touches = sum(abs(df['minimo'] - df['minimo'].iloc[i]) < 
                                df['minimo'].iloc[i] * self.data_config.price_cluster_threshold)
                    if touches >= self.data_config.min_touches:
                        levels.append({
                            'price': df['minimo'].iloc[i],
                            'volume': df['volume'].iloc[i],
                            'touches': touches,
                            'last_test': df['date'].iloc[i:].max(),
                            'type': 'swing_low',
                            'weight': 0.40  # Peso do método
                        })
            
            return levels
            
        except Exception as e:
            self.logger.error(f"Erro em níveis de swing: {str(e)}")
            return []
    
    def _find_gap_levels(self, df: pd.DataFrame) -> List[dict]:
        """Encontra níveis por Gaps"""
        levels = []
        try:
            df['gap'] = df['abertura'] - df['ultimo'].shift(1)
            df['gap_perc'] = df['gap'] / df['ultimo'].shift(1) * 100
            
            # Gaps significativos (>0.5%)
            significant_gaps = df[abs(df['gap_perc']) > 0.5]
            
            for _, gap in significant_gaps.iterrows():
                touches = sum(abs(df['ultimo'] - gap['abertura']) < 
                            gap['abertura'] * self.data_config.price_cluster_threshold)
                if touches >= self.data_config.min_touches:
                    levels.append({
                        'price': gap['abertura'],
                        'volume': gap['volume'],
                        'touches': touches,
                        'last_test': df[df['date'] >= gap['date']]['date'].max(),
                        'type': 'gap',
                        'weight': 0.30  # Peso do método
                    })
            
            return levels
            
        except Exception as e:
            self.logger.error(f"Erro em níveis de gap: {str(e)}")
            return []
    
    def _calculate_level_metrics(self, level: dict, current_price: float, 
                               avg_volume: float) -> dict:
        """Calcula métricas do nível"""
        try:
            # Fatores de força
            touch_factor = min(level['touches'] / self.data_config.min_touches, 3.0)
            volume_factor = min(level['volume'] / avg_volume, 2.0)
            
            # Distância do preço atual
            distance = abs(level['price'] - current_price)
            distance_perc = (distance / current_price) * 100
            
            # Recência do teste
            days_since_test = (pd.Timestamp.now() - level['last_test']).days
            recency_factor = 1 / (1 + days_since_test/30)
            
            # Força final (ponderada pelo tipo)
            strength = (
                touch_factor * 0.35 +      # Toques
                volume_factor * 0.25 +     # Volume
                (1 - distance_perc/5) * 0.25 +  # Proximidade
                recency_factor * 0.15      # Recência
            ) * level['weight']            # Peso do método
            
            return {
                'strength': strength,
                'distance': distance,
                'distance_perc': distance_perc
            }
            
        except Exception as e:
            self.logger.error(f"Erro no cálculo de métricas: {str(e)}")
            return {'strength': 0, 'distance': 0, 'distance_perc': 0}
    
    def _print_analysis_report(self, date: datetime, last_close: float,
                             supports: List[dict], resistances: List[dict]):
        """Imprime relatório de análise"""
        print("\n" + "="*60)
        print(f"ANÁLISE WDO - Próxima Sessão")
        print("="*60)
        
        print(f"\nÚltimo Fechamento: {last_close:,.2f}")
        
        # Filtro
        strong_resistances = sorted(
            [r for r in resistances if r['strength'] > 0.50],
            key=lambda x: (-x['strength'], x['distance'])
        )[:30]
        
        strong_supports = sorted(
            [s for s in supports if s['strength'] > 0.50],
            key=lambda x: (-x['strength'], x['distance'])
        )[:30]
        
        print(f"\nTOP RESISTÊNCIAS :")
        print("-"*60)
        for i, r in enumerate(strong_resistances, 1):
            print(f"{i}. Preço: {r['price']:,.2f} | "
                  f"Dist: {r['distance_perc']:,.2f}% | "
                  f"Força: {r['strength']:,.2f}")
        
        print(f"\nTOP SUPORTES :")
        print("-"*60)
        for i, s in enumerate(strong_supports, 1):
            print(f"{i}. Preço: {s['price']:,.2f} | "
                  f"Dist: {s['distance_perc']:,.2f}% | "
                  f"Força: {s['strength']:,.2f}")
        
        print("\n" + "="*60)
    
    def _save_analysis(self, date: datetime, last_close: float,
                      supports: List[dict], resistances: List[dict]):
        """Salva análise em arquivo"""
        analysis = {
            'date': date,
            'last_close': last_close,
            'supports': supports,
            'resistances': resistances,
            'timestamp': datetime.now()
        }
        
        file_path = os.path.join(
            self.ANALYSIS_PATH,
            f"wdo_analysis_{date.strftime('%Y%m%d')}.pkl"
        )
        
        with open(file_path, 'wb') as f:
            pickle.dump(analysis, f)

if __name__ == "__main__":
    # Configuração de caminhos
    BASE_PATH = "C:\ARQUIVOS\OneDrive\Bolsa\COLETA\B3\WDO"
    CACHE_PATH = "C:\ARQUIVOS\OneDrive\Bolsa\COLETA\B3\WDO\cache"
    
    try:
        # Inicializa análise
        wdo = WDOAnalysis(BASE_PATH, CACHE_PATH)
        
        # Executa análise para próxima sessão
        wdo.analyze_next_session()
        
    except Exception as e:
        print(f"Erro durante execução: {str(e)}")



2024-11-12 21:07:50,863 - INFO - 
Iniciando análise WDO para próxima sessão...
Carregando dados WDO: 100%|██████████| 1252/1252 [00:07<00:00, 175.76it/s]
2024-11-12 21:07:57,993 - INFO - Arquivos processados: 1252/1252
2024-11-12 21:07:58,053 - INFO - Registros carregados: 1252
2024-11-12 21:07:58,055 - INFO - Período: 2019-11-04 00:00:00 até 2024-11-12 00:00:00
2024-11-12 21:07:58,056 - INFO - Último preço: 5,758.00
2024-11-12 21:07:58,068 - INFO - Data base: 2024-11-12 00:00:00
2024-11-12 21:07:58,069 - INFO - Último fechamento: 5758.0
2024-11-12 21:07:58,072 - INFO - Processando 1252 registros
2024-11-12 21:07:58,072 - INFO - Último fechamento: 5758.0
2024-11-12 21:07:58,137 - INFO - Níveis por volume: 20
2024-11-12 21:07:58,233 - INFO - Níveis por swing: 43
2024-11-12 21:07:58,360 - INFO - Níveis por gap: 247
2024-11-12 21:07:58,379 - INFO - Suportes encontrados: 30
2024-11-12 21:07:58,380 - INFO - Resistências encontradas: 19



ANÁLISE WDO - Próxima Sessão

Último Fechamento: 5,758.00

TOP RESISTÊNCIAS :
------------------------------------------------------------
1. Preço: 5,762.00 | Dist: 0.07% | Força: 0.74
2. Preço: 5,883.00 | Dist: 2.17% | Força: 0.69
3. Preço: 5,814.00 | Dist: 0.97% | Força: 0.67
4. Preço: 5,882.50 | Dist: 2.16% | Força: 0.67
5. Preço: 5,768.00 | Dist: 0.17% | Força: 0.66
6. Preço: 5,868.00 | Dist: 1.91% | Força: 0.53
7. Preço: 5,840.00 | Dist: 1.42% | Força: 0.51
8. Preço: 5,785.00 | Dist: 0.47% | Força: 0.51
9. Preço: 5,786.00 | Dist: 0.49% | Força: 0.51
10. Preço: 5,787.00 | Dist: 0.50% | Força: 0.50

TOP SUPORTES :
------------------------------------------------------------
1. Preço: 5,717.00 | Dist: 0.71% | Força: 0.71
2. Preço: 5,674.50 | Dist: 1.45% | Força: 0.70
3. Preço: 5,548.00 | Dist: 3.65% | Força: 0.65
4. Preço: 5,527.00 | Dist: 4.01% | Força: 0.64
5. Preço: 5,520.00 | Dist: 4.13% | Força: 0.62
6. Preço: 5,509.50 | Dist: 4.32% | Força: 0.61
7. Preço: 5,465.50 | Dist: 5.0